# MobiHet P2 `p2-dml-s43`
Arms ['dml_dagsnet_B3_naive', 'dml_dagsnet_B2_naive', 'dml_dagsnet_B2_h6', 'dml_dagsnet_B0_h6'], seed 43. Plan: docs/research/pilot-plan.md §5.

In [ ]:
CODE_FILES = ["__init__.py", "pilot.py", "p2.py", "channel.py", "mobility.py", "dagsnet.py", "metrics.py", "archs.py"]

import json, os, shutil, subprocess, sys, time, hashlib
from pathlib import Path
import torch
n = torch.cuda.device_count()
print('torch', torch.__version__, 'gpus', [torch.cuda.get_device_name(i) for i in range(n)], flush=True)
assert n == 2, f'expected 2 GPUs, got {n}. Set machine_shape=NvidiaTeslaT4.'
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(n))
hits = sorted(p.parent for p in Path('/kaggle/input').rglob('p2.py') if (p.parent / 'train_0_X.npy').exists())
assert len(hits) == 1, hits
SRC, WORK, OUT = hits[0], Path('/kaggle/working/mobihet'), Path('/kaggle/working/runs')
CODE = CODE_FILES
for sub in ('mh', 'knowledge', 'papers/pilot/fixture'):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
for f in sorted(SRC.iterdir()):             # the dataset is flat: rebuild the repo layout the code expects
    dst = WORK / 'mh' / f.name if f.name in CODE else WORK / 'knowledge' / f.name if f.name == 'meta.json' else WORK / 'papers/pilot/fixture' / f.name
    shutil.copyfile(f, dst)
man = json.loads((WORK / 'papers/pilot/fixture/manifest.json').read_text())
for name, digest in man['files'].items():
    h = hashlib.sha256((WORK / 'papers/pilot/fixture' / name).read_bytes()).hexdigest()
    assert h == digest, ('fixture corrupted', name)
print('fixture verified:', len(man['files']), 'files', flush=True)


In [ ]:
SEED = 43
BUDGET_H = 9.0
JOBS = [["dml_dagsnet_B3_naive", "--method dml --proxy dagsnet --int8 --tier B3 --policy naive"], ["dml_dagsnet_B2_naive", "--method dml --proxy dagsnet --int8 --tier B2 --policy naive"], ["dml_dagsnet_B2_h6", "--method dml --proxy dagsnet --int8 --tier B2 --policy h6"], ["dml_dagsnet_B0_h6", "--method dml --proxy dagsnet --int8 --tier B0 --policy h6"]]

T0, slots, running, done = time.time(), [0, 0, 1, 1], {}, {}
OUT.mkdir(parents=True, exist_ok=True)
queue = [a for a in JOBS if not (OUT / f"{a[0]}_s{SEED}" / 'result.json').exists()]
seen = {}
while queue or running:
    for slot, gpu in enumerate(slots):
        if slot not in running and queue and time.time() - T0 < BUDGET_H * 3600:
            arm, args = queue.pop(0)
            d = OUT / f'{arm}_s{SEED}'; d.mkdir(parents=True, exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='1')
            cmd = [sys.executable, '-m', 'mh.p2', '--arm', arm, '--seed', str(SEED), '--out', str(OUT)] + args.split()
            running[slot] = (arm, subprocess.Popen(cmd, cwd=WORK, env=env, stdout=open(d / 'log.txt', 'w'), stderr=subprocess.STDOUT))
            print(time.strftime('%H:%M:%S'), 'start', arm, 'gpu', gpu, flush=True)
    for slot, (arm, p) in list(running.items()):
        log = OUT / f'{arm}_s{SEED}' / 'log.txt'
        lines = log.read_text().splitlines() if log.exists() else []
        for l in lines[seen.get(arm, 0):]:
            if l.startswith(('r', 'done', 'Traceback')) or 'Error' in l:
                print(arm, '|', l, flush=True)
        seen[arm] = len(lines)
        if p.poll() is not None:
            done[arm] = p.returncode; del running[slot]
            print(time.strftime('%H:%M:%S'), 'exit', arm, p.returncode, flush=True)
    if not running and queue and time.time() - T0 >= BUDGET_H * 3600:
        print('budget reached, not started:', [a for a, _ in queue], flush=True); break
    time.sleep(20)
summary = {a: dict(rc=rc, result=(OUT / f'{a}_s{SEED}' / 'result.json').exists()) for a, rc in done.items()}
(OUT / f'summary_s{SEED}.json').write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1), flush=True)
shutil.rmtree(WORK, ignore_errors=True)   # keep the output to runs/ only
assert all(v['rc'] == 0 and v['result'] for v in summary.values()), summary
